# Jumper on Kaggle — the second jump (`jumper.stairs_free`, two runs on two GPUs)

**Before anything else (Settings, right-hand bar):** Accelerator **GPU T4 x2** (never P100), Internet **On**, and under *Input* the Dataset `jumper-stairs-task` (the task's code, the two starting checkpoints and `tasks/jumper/stairs/landing_bank.npz`).

**How to use it.** Edit cell 1 only, then **Save Version → Save & Run All (Commit)**: the notebook runs in the background with the browser closed, up to 12 h, and at the end `/kaggle/working/jumper/` (backup ZIPs, curves, videos) is kept as the version's output. An interactive *Run All* is fine for a trial, but its output is lost unless downloaded from the Output panel.

**What it does.** Same setup as the official notebook (clone at the pinned commit, isolated venv, torch cu126, EGL, CUDA/Warp check) + the task's code from the Dataset over the clone → one starting checkpoint per run, converted with `tools/warm_start.py` (`--critic graft --keep-iter`: the actor and the critic are kept, the critic gets its one new input) → a short trial run → the long training, **one run per GPU, both the same task with different starts** (A from `model_10650`, B from `model_6750`), stopped by the clock after `TRAIN_HOURS` or earlier by the stop rules → curves, stairs metrics, replay MP4, backup ZIP, and a side-by-side table of the two runs.

**What to read in cell 4's log (one line a minute per run):** `level` (curriculum row, 0–9 → riser 2–15 cm), `steps_by_flight` (treads gained by jumping per episode), **`chained`** (share of episodes with two or more treads gained by jumping — the second jump), `rejumps`, `stall` (episodes ended for standing still), `fell`.

In [ ]:
# @title 1. Settings (edit this cell only)
import json
import math
import os
import re
import shutil
import signal
import subprocess
import sys
import threading
import time
import venv
import zipfile
from collections import deque
from datetime import datetime, timezone
from pathlib import Path

# ── The two runs: label, task, starting checkpoint (a file name in the Dataset) ──
# Same task and same code on both GPUs; only the start differs. A = the landing
# corrections (model_10650); B = the original C (model_6750), the control.
RUNS = [
    {"label": "A-10650", "task": "jumper.stairs_free", "start": "model_10650.pt"},
    {"label": "B-6750", "task": "jumper.stairs_free", "start": "model_6750.pt"},
]
NUM_ENVS = 2048                        # per process; a T4's 16 GB has room under the ~12 000 MiB peak
START_ROWS = "3-7"                     # curriculum rows the robots start on (the checkpoints carry none)

# ── Time ──
TRAIN_HOURS = 10.0      # training; then ~1 h for backup, curves, replay (Kaggle cuts at 12 h). For a trial: 0.5
SMOKE_ITERATIONS = 5    # short trial run before the long one (0 to skip)

# ── Stop rules, checked once per minute after STOP_AFTER_HOURS, on the mean of the last 20 iterations ──
STOP_AFTER_HOURS = 3.0
STOP_MIN_STEPS_BY_FLIGHT = 0.51   # C's level before this session: below it the run has lost the jump
STOP_FELL_FACTOR = 2.0            # fell_over above this times the run's own first-20-iteration mean
STOP_MIN_LEVEL = 3.0              # curriculum row below this: the run has slid back
STOP_ENABLED = True

# ── Sources ──
REPOSITORY_URL = "https://github.com/KingKongRobotics/jumper.git"
SOURCE_REVISION = "b2e58669848a3ba4f209597cd6bebdfe9d0533a4"   # pinned: the resume compares it
PATCH_DIR = "/kaggle/input/jumper-stairs-task"   # the Dataset with the task's code (zip or folder)

# ── Replay ──
REPLAY_SECONDS = 20.0   # final replay, per run
REPLAY_ROW = ""         # "" = the mean row the curriculum reached; or 0..9
REPLAY_WIDTH, REPLAY_HEIGHT, REPLAY_FPS = 960, 540, 30.0
# Camera: azimuth 90 = side view (0 = from behind, 180 = from the front), negative elevation looks down
REPLAY_AZIMUTH, REPLAY_ELEVATION, REPLAY_DISTANCE = 90.0, -8.0, 1.3
# Periodic replays during training (on the run's GPU, from the last complete
# checkpoint): the first after REPLAY_FIRST_HOURS, then every REPLAY_EVERY_HOURS. 0 = never.
REPLAY_EVERY_HOURS = 2.0
REPLAY_FIRST_HOURS = 0.5
REPLAY_PERIODIC_SECONDS = 10.0
REPLAY_MIN_FREE_MIB = 2500   # skipped when the GPU has less memory free than this

# ── Derived ──
ON_KAGGLE = Path("/kaggle").is_dir()
WORK = Path("/kaggle/temp") if ON_KAGGLE else Path("/tmp/jumper-work")     # ephemeral
OUT = Path("/kaggle/working/jumper") if ON_KAGGLE else Path.cwd() / "jumper-out"  # persistent (output)
REPO = WORK / "jumper"
PYTHON = REPO / ".venv/bin/python"
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
SESSION = OUT / f"session-{STAMP}"
BACKUPS = OUT / "backups"
LABELS = [r["label"] for r in RUNS]
RUN = {r["label"]: r for r in RUNS}
for directory in (WORK, OUT, SESSION, BACKUPS):
    directory.mkdir(parents=True, exist_ok=True)
if not (3, 10) <= sys.version_info[:2] <= (3, 13):
    raise RuntimeError(f"Jumper wants Python 3.10-3.13, this is {sys.version}")
if len(LABELS) != len(set(LABELS)):
    raise ValueError("run labels must be distinct")
for r in RUNS:
    if not re.fullmatch(r"jumper\.[a-z_]+", r["task"]):
        raise ValueError(f"not a task id: {r['task']}")
    if not re.fullmatch(r"[A-Za-z0-9_-]+", r["label"]):
        raise ValueError(f"label must be a file-name-safe word: {r['label']}")
print(f"{[(r['label'], r['task'], r['start']) for r in RUNS]} x {NUM_ENVS} envs, {TRAIN_HOURS} h, rows {START_ROWS}. Output in {OUT}")

In [ ]:
# @title 2. Setup: GPU, clone, venv, torch, EGL, the task's code  (~6-10 min)
def run(command, *, cwd=None, check=True, env=None, timeout_s=None, tail_lines=80):
    """Run and stream the output. Past `timeout_s`, SIGINT to the whole process
    group (training and helpers save and close) and the exit code comes back."""
    print("+", " ".join(str(p) for p in command), flush=True)
    command = [str(p) for p in command]
    tail = deque(maxlen=tail_lines)
    process = subprocess.Popen(
        command, cwd=cwd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1, start_new_session=True,
    )
    timer = None
    if timeout_s:
        timer = threading.Timer(timeout_s, lambda: os.killpg(process.pid, signal.SIGINT))
        timer.daemon = True
        timer.start()
    try:
        for line in process.stdout:
            print(line, end="", flush=True)
            tail.append(line[-4096:])
        code = process.wait()
        if code and check:
            raise subprocess.CalledProcessError(code, command, output="".join(tail))
        return subprocess.CompletedProcess(command, code, stdout="".join(tail))
    except BaseException:
        if process.poll() is None:
            try:
                os.killpg(process.pid, signal.SIGINT)
                process.wait(timeout=30)
            except (ProcessLookupError, subprocess.TimeoutExpired):
                try:
                    os.killpg(process.pid, signal.SIGKILL)
                except ProcessLookupError:
                    pass
        raise
    finally:
        if timer:
            timer.cancel()
        process.stdout.close()


# ── GPU ──
run(["nvidia-smi", "-L"])
rows = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,compute_cap,memory.total", "--format=csv,noheader"], text=True,
).strip().splitlines()
if not rows:
    raise RuntimeError("No GPU: Settings → Accelerator → GPU T4 x2")
gpus = [[p.strip() for p in r.split(",")] for r in rows]
capability = float(gpus[0][2])
if capability < 7.5:
    raise RuntimeError(f"GPU {gpus[0][1]} (compute capability {capability}) is not supported: choose T4 x2, not P100")
GPU_NAME = gpus[0][1]
if len(RUNS) > len(gpus):
    raise RuntimeError(f"{len(RUNS)} runs need {len(RUNS)} GPUs: choose T4 x2 or keep one run")
print("GPU:", [g[1] for g in gpus])

# ── Clone at the pinned commit ──
if not REPO.exists():
    run(["git", "clone", REPOSITORY_URL, REPO])
run(["git", "fetch", "origin", SOURCE_REVISION], cwd=REPO)
run(["git", "checkout", "--detach", SOURCE_REVISION], cwd=REPO)
COMMIT = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()

# ── The task's code, from the Dataset, over the clone ──
patch = Path(PATCH_DIR)
if not patch.exists():
    # Kaggle mounts datasets under /kaggle/input at a path that has changed over
    # time (<slug>, or datasets/<user>/<slug>): look for a landmark file.
    import glob as _glob
    hits = _glob.glob("/kaggle/input/**/tasks/jumper/stairs/env_cfg.py", recursive=True)
    if not hits:
        found = sorted(str(p) for p in Path("/kaggle/input").rglob("*") if p.is_dir())[:40]
        raise FileNotFoundError(f"the task's code was not found under /kaggle/input. Folders seen: {found}")
    patch = Path(hits[0]).parents[3]
    print("The task's code is in", patch)
zips = sorted(patch.rglob("*.zip"))
if zips:
    with zipfile.ZipFile(zips[-1]) as z:
        names = z.namelist()
        if any(n.startswith("/") or ".." in Path(n).parts for n in names):
            raise ValueError("zip with unsafe paths")
        z.extractall(REPO)
    print("Applied", zips[-1], f"({len(names)} files)")
else:
    copied = 0
    for src in patch.rglob("*"):
        if src.is_file():
            rel = src.relative_to(patch)
            # the dataset folder may carry one extra level (the dataset's name)
            parts = rel.parts
            if parts and parts[0] not in ("tasks", "tools", "notebooks", "docs", "checkpoints"):
                rel = Path(*parts[1:])
            dst = REPO / rel
            dst.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(src, dst)
            copied += 1
    print(f"Copied {copied} files from {patch}")
for required in ("tasks/jumper/stairs/env_cfg.py", "tasks/jumper/stairs/landing_bank.npz",
                 "tools/warm_start.py", "tools/stairs_tremor.py"):
    if not (REPO / required).is_file():
        raise FileNotFoundError(f"{required} is missing: the Dataset does not hold the task's code")

# ── The starting checkpoints, by file name, anywhere under the Dataset or the clone ──
START_CKPT = {}
for r in RUNS:
    hits = sorted(set(list(patch.rglob(r["start"])) + list((REPO / "checkpoints").rglob(r["start"]))))
    if not hits:
        raise FileNotFoundError(f"{r['label']}: {r['start']} not found under {patch} (add it to the Dataset)")
    START_CKPT[r["label"]] = hits[0]
    print(f"{r['label']}: starts from {hits[0]} ({hits[0].stat().st_size / 1e6:.1f} MB)")

# ── Isolated venv ──
if not PYTHON.exists():
    venv.EnvBuilder(with_pip=False, system_site_packages=False).create(REPO / ".venv")
run([sys.executable, "-m", "pip", "--python", PYTHON, "install", "--upgrade", "pip", "-q"])
try:
    run(["apt-get", "install", "-y", "-qq", "libegl1", "libgl1", "ffmpeg"], check=False)
except Exception as exc:  # noqa: BLE001
    print("apt-get failed (continuing):", exc)
TORCH_INDEX = "cu126" if capability < 8.0 else "cu128"
run([PYTHON, "-m", "pip", "install", "-q", "torch==2.9.1", "torchvision==0.24.1",
     "--index-url", f"https://download.pytorch.org/whl/{TORCH_INDEX}"])
run([PYTHON, "-m", "pip", "install", "-q", "-e", str(REPO)])
# Note: `pip install -e .` may resolve torch past 2.9.1 (measured: 2.14.1 from
# PyPI on 2026-10-27); the versions actually installed go into runtime.json.
run([PYTHON, "-m", "pip", "install", "-q", "matplotlib"])

# ── Headless rendering for the replay MP4 (training does not need it) ──
# The official notebook requires NVIDIA's EGL; Kaggle's image does not expose it
# (measured: no libEGL_nvidia on the standard paths). Look everywhere and, failing
# that, fall back to Mesa's software EGL: enough for a short single-robot video.
# The replay is in try/except anyway: it never blocks the backup.
import glob as _glob
os.environ.update(MUJOCO_GL="egl", PYTHONUNBUFFERED="1", MPLBACKEND="Agg", PYOPENGL_PLATFORM="egl")
_egl_hits = []
for pattern in ("/usr/lib64-nvidia/libEGL_nvidia.so.*", "/usr/lib/x86_64-linux-gnu/libEGL_nvidia.so.*",
                "/usr/lib/x86_64-linux-gnu/nvidia/current/libEGL_nvidia.so.*", "/usr/lib/nvidia/libEGL_nvidia.so.*",
                "/usr/local/nvidia/lib64/libEGL_nvidia.so.*", "/usr/local/cuda*/lib64/libEGL_nvidia.so.*"):
    _egl_hits += [h for h in _glob.glob(pattern) if Path(h).is_file()]
if not _egl_hits:
    try:
        _egl_hits = subprocess.run(["bash", "-lc", "find / -xdev -name 'libEGL_nvidia.so.*' 2>/dev/null | head -5"],
                                   capture_output=True, text=True, timeout=120).stdout.split()
    except Exception:  # noqa: BLE001
        _egl_hits = []
if _egl_hits:
    lib = Path(_egl_hits[0]).resolve()
    (WORK / "nvidia-egl.json").write_text(json.dumps({"file_format_version": "1.0.0", "ICD": {"library_path": str(lib)}}))
    os.environ["__EGL_VENDOR_LIBRARY_FILENAMES"] = str(WORK / "nvidia-egl.json")
    os.environ["LD_LIBRARY_PATH"] = ":".join([str(lib.parent), *[e for e in os.environ.get("LD_LIBRARY_PATH", "").split(":") if e]])
    print("NVIDIA EGL:", lib)
else:
    print("No NVIDIA EGL: the replay will use Mesa's software EGL (slower, but enough for a short video)")
    try:
        run(["apt-get", "install", "-y", "-qq", "libegl-mesa0", "libgl1-mesa-dri", "libgles2"], check=False)
    except Exception as exc:  # noqa: BLE001
        print("apt-get mesa failed (continuing):", exc)
    os.environ["EGL_PLATFORM"] = "surfaceless"

run([PYTHON, "tools/colab.py", "check", "--repo", REPO, "--out", SESSION / "runtime.json"], cwd=REPO)
runtime = json.loads((SESSION / "runtime.json").read_text())
listing = run([PYTHON, "scripts/train.py", "--list"], cwd=REPO).stdout
for r in RUNS:
    if r["task"] not in listing:
        raise RuntimeError(f"{r['task']} is not in --list: the import line in tasks/__init__.py is missing from the Dataset")
run([PYTHON, "scripts/train.py", "--task", RUNS[0]["task"], "--dry-run"], cwd=REPO)
print("\nSETUP OK —", GPU_NAME, "| commit", COMMIT[:9], "| versions", runtime["versions"])

In [ ]:
# @title 3. Starting points: each run's checkpoint, converted to this task's current shape
START_FROM = {}   # label -> checkpoint `tools/colab.py train --resume` starts from

# `tools/colab.py train --resume` wants the checkpoint's manifest beside it, with
# the configuration and the file's sha; this helper writes it with colab.py's own
# functions, so the compatibility fields are exactly its own.
manifest_helper = r"""
import importlib.util, json, sys, uuid
from pathlib import Path
repo, task, model, num_envs, ckpt = Path(sys.argv[1]), sys.argv[2], sys.argv[3], int(sys.argv[4]), Path(sys.argv[5])
spec = importlib.util.spec_from_file_location("colab", repo / "tools/colab.py")
colab = importlib.util.module_from_spec(spec); spec.loader.exec_module(colab)
cfg = colab.make_config(repo, task=task, model=model, num_envs=num_envs)
manifest = {"schema": colab.SCHEMA, "operation": "warm-" + uuid.uuid4().hex, "config": cfg,
            "command": ["tools/warm_start.py"], "status": "finished",
            "checkpoints": {ckpt.name: colab.sha256(ckpt)}, "checkpoint": str(ckpt),
            "resume_from": None, "run_directory": str(ckpt.parent), "previous_manifest": None}
colab.atomic_json(ckpt.parent / colab.MANIFEST, manifest)
colab.validate_resume(ckpt, cfg)
print("manifest ok:", ckpt.parent / colab.MANIFEST)
"""

for r in RUNS:
    label, task, src = r["label"], r["task"], START_CKPT[r["label"]]
    out = REPO / "logs" / "warm" / label / src.name
    # The critic grew one input (the stall clock) on 2026-11-04; the actor did
    # not. `--critic graft` keeps both and gives the critic its fresh column;
    # `--keep-iter` continues the iteration count so the curves join up.
    run([PYTHON, "tools/warm_start.py", "--task", task, "--from", src, "--out", out,
         "--critic", "graft", "--keep-iter", "--num-envs", "4", "--device", "cpu"], cwd=REPO)
    run([PYTHON, "-c", manifest_helper, REPO, task, "jumper", str(NUM_ENVS), out], cwd=REPO)
    START_FROM[label] = str(out)
    print(label, "starts from", out)
print("Starts:", START_FROM)

In [ ]:
# @title 4. Trial run, then the long training (one run per GPU, stopped by the clock or by the rules)
def train_env(gpu_index):
    """Each process sees one GPU, as `cuda:0`: colab.py always trains on cuda:0.
    The stairs' start rows go in by environment variable (`mdp/curriculum.py`)."""
    e = os.environ.copy()
    e["CUDA_VISIBLE_DEVICES"] = str(gpu_index)
    e["MJRL_STAIRS_START_ROWS"] = START_ROWS
    return e


def train_command(task, iterations, manifest, num_envs, resume=None):
    cmd = [str(PYTHON), "tools/colab.py", "train", "--repo", str(REPO), "--task", task, "--model", "jumper",
           "--num-envs", str(num_envs), "--iterations", str(iterations), "--manifest", str(manifest)]
    if resume:
        cmd += ["--resume", str(resume)]
    return cmd


# ── Trial run: if the scene does not build (nconmax, a term that does not resolve,
# a checkpoint that does not load) it must fail here, in minutes, not after 10 h. ──
if SMOKE_ITERATIONS > 0:
    smoke_procs = {}
    for gpu, label in enumerate(LABELS):
        smoke_manifest = SESSION / f"smoke-{label}.json"
        log = (SESSION / f"smoke-{label}.log").open("w")
        # Without --resume: the start's manifest is written for NUM_ENVS and colab.py
        # refuses another count. The checkpoint load shows in the long run's first minutes.
        smoke_procs[label] = (subprocess.Popen(train_command(RUN[label]["task"], SMOKE_ITERATIONS, smoke_manifest, min(NUM_ENVS, 256)),
                                               cwd=REPO, env=train_env(gpu), stdout=log, stderr=subprocess.STDOUT,
                                               text=True, bufsize=1, start_new_session=True), log, smoke_manifest)
    for label, (p, log, smoke_manifest) in smoke_procs.items():
        try:
            p.wait(timeout=900)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        log.close()
        smoke = json.loads(smoke_manifest.read_text()) if smoke_manifest.is_file() else {"status": "missing"}
        if smoke.get("status") != "finished":
            print((SESSION / f"smoke-{label}.log").read_text(errors="replace")[-3000:])
            raise RuntimeError(f"the trial run of {label} did not finish: {smoke.get('status')} {smoke.get('error')}")
        print(f"Trial run OK ({label}):", smoke["run_directory"])

# ── Periodic replays: 10 s of video from the last complete checkpoint, without stopping the training ──
RATES = {}
for r in RUNS:
    t = r["task"]
    if t in RATES:
        continue
    rf = SESSION / f"rates-{t}.json"
    try:
        run([PYTHON, "tools/colab.py", "rates", "--repo", REPO, "--task", t, "--model", "jumper", "--out", rf], cwd=REPO)
        RATES[t] = json.loads(rf.read_text())
    except Exception as exc:  # noqa: BLE001 -- without the rates no replay, but the training starts anyway
        print(f"[replay {t}] rates not read ({exc}): replays off for this task")
REPLAYS = SESSION / "replays"
REPLAYS.mkdir(exist_ok=True)
replay_locks = {label: threading.Lock() for label in LABELS}


def gpu_free_mib(gpu):
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.free", "--format=csv,noheader,nounits",
                                       "-i", str(gpu)], text=True)
        return float(out.strip().splitlines()[0])
    except Exception:  # noqa: BLE001
        return 0.0


def latest_complete_checkpoint(manifest_path):
    """The last model_N.pt colab.py has recorded (so complete, with its sha)."""
    m = json.loads(manifest_path.read_text())
    run_dir, recorded = m.get("run_directory"), m.get("checkpoints") or {}
    if not run_dir or not recorded:
        return None
    name = max(recorded, key=lambda n: int(re.search(r"model_(\d+)", n).group(1)))
    return Path(run_dir) / name


def replay_command(task, checkpoint, video, seconds, row, extra=()):
    steps = math.ceil(seconds * RATES[task]["control_hz"])
    cmd = [PYTHON, "scripts/play.py", "--task", task, "--model", "jumper", "--checkpoint", checkpoint,
           "--backend", "warp", "--device", "cuda:0", "--num_envs", "1", "--headless", "--steps", steps,
           "--physics-hz", RATES[task]["physics_hz"], "--video", video, "--video-fps", REPLAY_FPS,
           "--video-width", REPLAY_WIDTH, "--video-height", REPLAY_HEIGHT, "--strip-visual", "off",
           "--video-azimuth", REPLAY_AZIMUTH, "--video-elevation", REPLAY_ELEVATION,
           "--stairs-row", row, *extra]
    if REPLAY_DISTANCE:
        cmd += ["--video-distance", REPLAY_DISTANCE]
    return [str(c) for c in cmd]


def periodic_replay(label, gpu, row):
    task = RUN[label]["task"]
    if not replay_locks[label].acquire(blocking=False):
        return
    try:
        if task not in RATES:
            return
        ckpt = latest_complete_checkpoint(MANIFESTS[label])
        if ckpt is None or not ckpt.is_file():
            print(f"[replay {label}] no complete checkpoint yet", flush=True)
            return
        free = gpu_free_mib(gpu)
        if free < REPLAY_MIN_FREE_MIB:
            print(f"[replay {label}] skipped: {free:.0f} MiB free on GPU {gpu}", flush=True)
            return
        it = re.search(r"model_(\d+)", ckpt.name).group(1)
        video = REPLAYS / f"{label}-it{it}-row{row}.mp4"
        r = subprocess.run(replay_command(task, ckpt, video, REPLAY_PERIODIC_SECONDS, row), cwd=REPO,
                           env=train_env(gpu), capture_output=True, text=True, timeout=1500)
        if r.returncode == 0 and video.is_file():
            print(f"[replay {label}] {video.relative_to(OUT)} ({video.stat().st_size / 1e6:.1f} MB)", flush=True)
        else:
            print(f"[replay {label}] failed ({r.returncode}):\n{r.stdout[-1500:]}", flush=True)
    except Exception as exc:  # noqa: BLE001 -- a replay never stops the training
        print(f"[replay {label}] error: {exc}", flush=True)
    finally:
        replay_locks[label].release()


# ── Reading the log: the last value of each key, and the mean of the last 20 iterations ──
KEYS = {
    "it": r"Learning iteration (\d+)/", "reward": r"Mean reward:\s*([-0-9.]+)",
    "level": r"Curriculum/stairs/level:\s*([-0-9.]+)", "steps_by_flight": r"steps_by_flight:\s*([-0-9.]+)",
    "chained": r"chained:\s*([-0-9.]+)", "rejumps": r"rejumps:\s*([-0-9.]+)",
    "crawled": r"steps_crawled:\s*([-0-9.]+)", "fell": r"Episode_Termination/fell_over:\s*([-0-9.]+)",
    "stall": r"Episode_Termination/stall:\s*([-0-9.]+)",
}


def read_log(label):
    text = (SESSION / f"train-{label}.log").read_text(errors="replace")
    out = {}
    for key, pattern in KEYS.items():
        values = re.findall(pattern, text)
        out[key] = [float(v) for v in values]
    return out


def tail_mean(values, n=20):
    return sum(values[-n:]) / len(values[-n:]) if values else None


# ── The long training ──
MANIFESTS = {label: SESSION / f"training-{label}.json" for label in LABELS}
procs = {}
t0 = time.time()
next_replay = t0 + REPLAY_FIRST_HOURS * 3600 if REPLAY_EVERY_HOURS > 0 else None
for gpu, label in enumerate(LABELS):
    cmd = train_command(RUN[label]["task"], 1_000_000, MANIFESTS[label], NUM_ENVS, START_FROM.get(label))
    print("+", " ".join(cmd), f"(GPU {gpu}, {label})", flush=True)
    log = (SESSION / f"train-{label}.log").open("w")
    procs[label] = (subprocess.Popen(cmd, cwd=REPO, env=train_env(gpu), stdout=log, stderr=subprocess.STDOUT,
                                     text=True, bufsize=1, start_new_session=True), log)

deadline = t0 + TRAIN_HOURS * 3600
stop_check = t0 + STOP_AFTER_HOURS * 3600
stopped = False
stopped_runs = {}
fell_baseline = {}
try:
    while any(p.poll() is None for p, _ in procs.values()):
        time.sleep(60)
        now = time.time()
        status = []
        for label, (p, log) in procs.items():
            v = read_log(label)
            if label not in fell_baseline and len(v["fell"]) >= 20:
                fell_baseline[label] = sum(v["fell"][:20]) / 20.0
            last = {k: (vals[-1] if vals else None) for k, vals in v.items()}
            fmt = lambda x, d=3: "-" if x is None else f"{x:.{d}f}"  # noqa: E731
            status.append(f"{label}: it {int(last['it']) if last['it'] is not None else '-'} reward {fmt(last['reward'], 2)}"
                          f" level {fmt(last['level'], 2)} steps_by_flight {fmt(last['steps_by_flight'])}"
                          f" chained {fmt(last['chained'])} rejumps {fmt(last['rejumps'], 2)}"
                          f" stall {fmt(last['stall'])} fell {fmt(last['fell'])}"
                          f"{'' if p.poll() is None else f' [exited {p.returncode}]'}")
        print(f"[{(now - t0) / 3600:5.2f} h] " + " | ".join(status), flush=True)
        if next_replay and now >= next_replay and not stopped:
            next_replay = now + REPLAY_EVERY_HOURS * 3600
            for gpu, label in enumerate(LABELS):
                if procs[label][0].poll() is None:
                    lv = read_log(label)["level"]
                    row = max(0, min(9, int(round(lv[-1])))) if lv else 0
                    threading.Thread(target=periodic_replay, args=(label, gpu, row), daemon=True).start()
        # ── The stop rules, per run, after STOP_AFTER_HOURS ──
        if STOP_ENABLED and now >= stop_check and not stopped:
            for label, (p, log) in procs.items():
                if p.poll() is not None or label in stopped_runs:
                    continue
                v = read_log(label)
                reasons = []
                sbf, lvl, fell = tail_mean(v["steps_by_flight"]), tail_mean(v["level"]), tail_mean(v["fell"])
                if sbf is not None and sbf < STOP_MIN_STEPS_BY_FLIGHT:
                    reasons.append(f"steps_by_flight {sbf:.3f} < {STOP_MIN_STEPS_BY_FLIGHT}")
                if lvl is not None and lvl < STOP_MIN_LEVEL:
                    reasons.append(f"level {lvl:.2f} < {STOP_MIN_LEVEL}")
                base = fell_baseline.get(label)
                if fell is not None and base is not None and base > 0 and fell > STOP_FELL_FACTOR * base:
                    reasons.append(f"fell_over {fell:.3f} > {STOP_FELL_FACTOR} x {base:.3f}")
                if reasons:
                    print(f">>> {label} stopped by the rules at {(now - t0) / 3600:.2f} h: " + "; ".join(reasons), flush=True)
                    stopped_runs[label] = {"hours": (now - t0) / 3600, "reasons": reasons}
                    (SESSION / f"stop-{label}.json").write_text(json.dumps(stopped_runs[label], indent=2))
                    os.killpg(p.pid, signal.SIGINT)
            stop_check = now + 3600  # the rules are checked again every hour
        if now >= deadline and not stopped:
            print(">>> time budget spent: SIGINT (the latest checkpoint is already on disk)", flush=True)
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGINT)
            stopped = True
        if stopped and now >= deadline + 300:
            for p, _ in procs.values():
                if p.poll() is None:
                    os.killpg(p.pid, signal.SIGKILL)
finally:
    for p, log in procs.values():
        try:
            p.wait(timeout=120)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid, signal.SIGKILL)
        log.close()

TRAINING = {}
RUN_DIR = {}
CHECKPOINT = {}
for label in list(LABELS):
    m = json.loads(MANIFESTS[label].read_text())
    print(f"{label}: status {m['status']}, run {m.get('run_directory')}, checkpoint {m.get('checkpoint')}")
    if m["status"] not in ("finished", "interrupted") or not m.get("checkpoint"):
        # The other run still has its hours: carry on without this one.
        print((SESSION / f"train-{label}.log").read_text(errors="replace")[-3000:])
        print(f"!!! {label}: training {m['status']}: {m.get('error')} -- left out of curves, replay and backup")
        LABELS.remove(label)
        continue
    TRAINING[label] = m
    RUN_DIR[label] = Path(m["run_directory"])
    CHECKPOINT[label] = Path(m["checkpoint"])
if not LABELS:
    raise RuntimeError("no run produced a checkpoint")
print(f"Training closed after {(time.time() - t0) / 3600:.2f} h; stopped by the rules: {stopped_runs or 'none'}")

In [ ]:
# @title 5. Curves, stairs metrics, replay MP4, backup ZIP, and A against B
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Video, display

ARTIFACTS = {}
KEY_TAGS = (
    "Metrics/stairs/steps_by_flight", "Metrics/stairs/chained", "Metrics/stairs/rejumps",
    "Metrics/stairs/steps_crawled", "Metrics/stairs/crawl_ratio",
    "Metrics/stairs/jumps", "Metrics/stairs/progress_m", "Metrics/stairs/apex_rise", "Metrics/stairs/flight_s", "Metrics/stairs/max_step",
    "Metrics/stairs/reached_top", "Metrics/stairs/edge_s", "Metrics/stairs/landing_dx_m",
    "Curriculum/stairs/level", "Curriculum/stairs/level_max",
    "Curriculum/stairs/riser_m", "Curriculum/stairs/promoted", "Curriculum/stairs/recent",
    "Episode_Termination/fell_over", "Episode_Termination/fell_off", "Episode_Termination/off_side",
    "Episode_Termination/stall", "Episode_Termination/reached_top",
    "Episode_Reward/illegal_contact", "Episode_Reward/crawl", "Episode_Reward/climbed_stream",
    "Episode_Reward/step_up_in_flight", "Train/mean_reward", "Policy/mean_noise_std",
)


def finite(points):
    return [(p["step"], p["value"]) for p in points if isinstance(p["value"], float)]


SUMMARY = {}
for label in LABELS:
    t = RUN[label]["task"]
    art = SESSION / "artifacts" / label
    art.mkdir(parents=True, exist_ok=True)
    ARTIFACTS[label] = art
    for name in ("runtime.json",):
        shutil.copy2(SESSION / name, art / name)
    for name in (f"stop-{label}.json",):
        if (SESSION / name).is_file():
            shutil.copy2(SESSION / name, art / name)

    # ── Curves: every scalar (official JSON+CSV+PNG) and a stairs panel ──
    metrics_json = art / "metrics.json"
    run([PYTHON, "tools/colab.py", "metrics", "--run", RUN_DIR[label], "--out", metrics_json,
         "--plot", art / "training-curves.png"], cwd=REPO)
    tags = json.loads(metrics_json.read_text())["tags"]
    present = [k for k in KEY_TAGS if k in tags and finite(tags[k])]
    if present:
        cols = 3
        rows = math.ceil(len(present) / cols)
        fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 2.8 * rows), squeeze=False)
        for ax, key in zip(axes.flat, present, strict=False):
            xs, ys = zip(*finite(tags[key]), strict=False)
            ax.plot(xs, ys, lw=1)
            ax.set_title(key, fontsize=9)
            ax.grid(alpha=0.3)
        for ax in list(axes.flat)[len(present):]:
            ax.axis("off")
        fig.suptitle(f"{label} ({t}) — {RUN_DIR[label].name}")
        fig.tight_layout()
        fig.savefig(art / "stairs-curves.png", dpi=110)
        plt.close(fig)
        display(Image(filename=str(art / "stairs-curves.png")))
    summary = {}
    for key in KEY_TAGS:
        pts = finite(tags.get(key, []))
        if pts:
            first = [v for _, v in pts[:20]]
            last = pts[-1]
            tail = [v for _, v in pts[-20:]]
            summary[key] = {"first_step": pts[0][0], "mean_first_20": sum(first) / len(first),
                            "last_step": last[0], "last": last[1], "mean_last_20": sum(tail) / len(tail)}
    SUMMARY[label] = summary
    (art / "stairs-summary.json").write_text(json.dumps(summary, indent=2))
    print(f"\n=== {label} ({t}): last 20 iterations (mean) ===")
    for key, v in summary.items():
        print(f"  {key:<42} {v['mean_last_20']:>10.4f}   (last {v['last']:.4f} @ it {v['last_step']}; first 20: {v['mean_first_20']:.4f})")

    # ── Replay MP4: the row the curriculum reached, from the landing ──
    level = summary.get("Curriculum/stairs/level", {}).get("mean_last_20", 0.0)
    row = REPLAY_ROW if REPLAY_ROW != "" else str(max(0, min(9, int(round(level)))))
    if REPLAYS.is_dir() and any(REPLAYS.iterdir()):
        shutil.copytree(REPLAYS, art / "replays", dirs_exist_ok=True)
    video = art / f"replay-{label}-row{row}.mp4"
    try:
        run(replay_command(t, CHECKPOINT[label], video, REPLAY_SECONDS, row,
                           extra=("--measure", "--replay-info-out", art / "replay-info.json")),
            cwd=REPO, env=train_env(0))
        display(Video(str(video), embed=True))
        for name in ("measure.csv", "measure.png"):
            src = CHECKPOINT[label].parent / "measure" / name
            if src.is_file():
                shutil.copy2(src, art / name)
    except Exception as exc:  # noqa: BLE001 -- the replay must not prevent the backup
        print(f"[replay {label}] failed: {exc}")
        video = None

    # ── Backup ZIP: checkpoint + manifest + curves + replay, in /kaggle/working ──
    backup = BACKUPS / f"{label}-{STAMP}.zip"
    cmd = [PYTHON, "tools/colab.py", "backup", "--run", RUN_DIR[label], "--out", backup, "--extras", art]
    if video and video.is_file():
        cmd += ["--video", video]
    run(cmd, cwd=REPO)
    print(f"BACKUP {label}: {backup} ({backup.stat().st_size / 1e6:.0f} MB)")

# ── A against B: the second jump, side by side ──
COMPARE = ("Metrics/stairs/chained", "Metrics/stairs/rejumps", "Metrics/stairs/steps_by_flight",
           "Metrics/stairs/jumps", "Metrics/stairs/apex_rise", "Curriculum/stairs/level",
           "Curriculum/stairs/level_max", "Episode_Termination/fell_over", "Episode_Termination/stall",
           "Metrics/stairs/edge_s", "Metrics/stairs/crawl_ratio", "Train/mean_reward", "Policy/mean_noise_std")
print("\n=== A against B (mean of the last 20 iterations; first 20 in brackets) ===")
print(f"{'key':<36}" + "".join(f"{label:>24}" for label in LABELS))
for key in COMPARE:
    cells = []
    for label in LABELS:
        v = SUMMARY[label].get(key)
        cells.append(f"{v['mean_last_20']:>12.4f} ({v['mean_first_20']:.3f})" if v else f"{'-':>24}")
    print(f"{key:<36}" + "".join(f"{c:>24}" for c in cells))
(SESSION / "ab-summary.json").write_text(json.dumps({label: SUMMARY[label] for label in LABELS}, indent=2))
print("\nDONE. Now: Save Version → Save & Run All (Commit) keeps /kaggle/working as the output.")

## If something goes wrong

- **`nconmax overflow (nconmax must be >= N)`** in the trial run: in the Dataset, in `tasks/jumper/stairs/env_cfg.py`, set `cfg.sim.nconmax` to exactly N and run again.
- **`cannot resume from ...`** at the start of the long run: the converted checkpoint does not match the network (observations changed). Report it; `tools/warm_start.py` is what converts the start.
- **`<label>: <model>.pt not found under /kaggle/input`**: the Dataset must hold the two checkpoints (any folder) beside the task's code.
- **Session cut before cell 5**: the checkpoints are in `/kaggle/temp` and are lost. The long run saves every 50 iterations, but only cell 5 backs them up: keep `TRAIN_HOURS` ≤ 10 so that an hour is left to close.
- **A run stopped by the rules**: its `stop-<label>.json` says why; the other run goes on to the end. The stopped run still gets its curves, replay and backup.
- **The replay fails** but the backup is there: fine, the video can be made again in another session from the backup.